# MNIST Digit Classification with Neural Network

A Multi-Layer Perceptron (MLP) model for classifying handwritten digits (0-9) from the MNIST dataset.

**Architecture:** Input (28x28) → Flatten → Dense(128, ReLU) → Dense(128, ReLU) → Dense(10, Softmax)

**Expected accuracy: ~97-98%** on the test set.

In [ ]:
import os, datetime, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## 1. Reproducibility

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

## 2. Load MNIST Dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Shapes:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test :", x_test.shape)
print("y_test :", y_test.shape)

## 3. Visualize Sample Images

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.suptitle("Sample MNIST Images", y=1.02)
plt.tight_layout()
plt.show()

## 4. Preprocess Data

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Pixel range after normalization:", x_train.min(), "to", x_train.max())

## 5. Build the Model

MLP with 2 hidden layers of 128 neurons each.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28), name="input_image"),
    layers.Flatten(name="flatten_layer"),
    layers.Dense(128, activation="relu", name="hidden_layer_1"),
    layers.Dense(128, activation="relu", name="hidden_layer_2"),
    layers.Dense(10, activation="softmax", name="output_layer")
], name="mnist_mlp")

print("MODEL SUMMARY")
model.summary()

## 6. Compile the Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 7. Setup Callbacks

In [ ]:
log_dir = os.path.join("logs", "fit", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

callbacks = []

try:
    import tensorboard  # noqa: F401
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=1
    )
    callbacks.append(tensorboard_callback)
    print("TensorBoard log directory:", log_dir)
    print("To view TensorBoard, run: tensorboard --logdir logs")
except ImportError:
    print("TensorBoard not installed, skipping TensorBoard callback.")

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)
callbacks.append(early_stopping_callback)

## 8. Train the Model

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=35,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

## 9. Plot Training History

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train_accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 10. Evaluate on Test Set

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")

## 11. Generate Predictions

In [ ]:
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

## 12. Classification Metrics

- **Accuracy**: proportion of total correct predictions
- **Precision**: among predicted samples of a class, how many are correct
- **Recall**: among real samples of a class, how many were found
- **F1-score**: harmonic mean balancing precision and recall
- **Macro avg**: simple average over all classes (treats all classes equally)
- **Weighted avg**: average weighted by class support (accounts for class imbalance)

In [ ]:
acc = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average="macro")
recall_macro = recall_score(y_test, y_pred, average="macro")
f1_macro = f1_score(y_test, y_pred, average="macro")

precision_weighted = precision_score(y_test, y_pred, average="weighted")
recall_weighted = recall_score(y_test, y_pred, average="weighted")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision (macro)",
        "Recall (macro)",
        "F1-score (macro)",
        "Precision (weighted)",
        "Recall (weighted)",
        "F1-score (weighted)"
    ],
    "Value": [
        acc,
        precision_macro,
        recall_macro,
        f1_macro,
        precision_weighted,
        recall_weighted,
        f1_weighted
    ]
})

print(metrics_df.to_string(index=False))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_pred, digits=4))

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Confusion Matrix (Counts)")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")

cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Oranges", ax=axes[1])
axes[1].set_title("Normalized Confusion Matrix")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")

plt.tight_layout()
plt.show()

## 14. Misclassified Examples

In [ ]:
misclassified_idx = np.where(y_pred != y_test)[0]
print(f"Total misclassified: {len(misclassified_idx)} out of {len(y_test)} ({len(misclassified_idx)/len(y_test)*100:.2f}%)")

plt.figure(figsize=(12, 8))
for i, idx in enumerate(misclassified_idx[:12]):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[idx], cmap="gray")
    conf = y_prob[idx][y_pred[idx]]
    plt.title(f"True: {y_test[idx]} | Pred: {y_pred[idx]}\nConf: {conf:.2%}")
    plt.axis("off")
plt.suptitle("Misclassified Digits", y=1.02)
plt.tight_layout()
plt.show()

## 15. Predict on a Custom Image (Optional)

Load a 28x28 grayscale image of a handwritten digit to test the model.

In [ ]:
import io, base64
from PIL import Image

def predict_from_image(image_path, model):
    """Load an image file, preprocess it, and predict the digit."""
    image = Image.open(image_path).convert("L")
    image = image.resize((28, 28))
    img_array = np.array(image).astype("float32") / 255.0
    img_input = np.expand_dims(img_array, axis=0)

    probs = model.predict(img_input, verbose=0)[0]
    pred_digit = int(np.argmax(probs))
    confidence = float(np.max(probs))

    plt.figure(figsize=(10, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img_array, cmap="gray")
    plt.title("Input image (28x28)")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    colors = ["steelblue"] * 10
    colors[pred_digit] = "tomato"
    plt.bar(range(10), probs, color=colors)
    plt.xticks(range(10))
    plt.xlabel("Digit")
    plt.ylabel("Probability")
    plt.title(f"Prediction: {pred_digit} (confidence: {confidence:.2%})")
    plt.tight_layout()
    plt.show()

    return pred_digit, confidence

# Uncomment and set the path to test with your own image:
# pred, conf = predict_from_image("my_digit.png", model)
print("To test with a custom image, call: predict_from_image('path/to/image.png', model)")

## 16. Save the Model

In [ ]:
model.save("mnist_mlp_model.keras")
print("Model saved as mnist_mlp_model.keras")

## 17. Accuracy Analysis & Potential Improvements

### Why does this model achieve ~97-98% accuracy?

This simple MLP (Multi-Layer Perceptron) with two 128-neuron hidden layers typically reaches
**97-98% test accuracy** on MNIST. Here's why, and how to go higher:

| Factor | Current Approach | Impact |
|--------|-----------------|--------|
| Architecture | Flatten → Dense(128) → Dense(128) → Dense(10) | The flatten operation discards spatial/2D structure of the image |
| Parameters | ~118K trainable parameters | Sufficient for MNIST but limited capacity |
| Regularization | None (no Dropout, no BatchNorm) | Some overfitting risk; early stopping helps |
| Data augmentation | None | Model only sees the original training samples |

### How to push beyond 99%

1. **Use a CNN** instead of MLP: Convolutional layers preserve spatial structure and achieve 99.2%+ with a simple LeNet-style architecture.

2. **Add Dropout** (e.g., `Dropout(0.25)` after each hidden layer) to reduce overfitting.

3. **Add Batch Normalization** (`BatchNormalization()`) after dense layers for faster, more stable training.

4. **Data augmentation**: Random rotations (±10°), shifts, and zoom create more training variety.

5. **Learning rate scheduling**: Reduce learning rate when validation loss plateaus.

6. **Increase model capacity**: More layers or wider layers (256, 512 neurons).

### Benchmark reference

| Model | Typical MNIST Accuracy |
|-------|----------------------|
| Simple MLP (this notebook) | 97-98% |
| MLP + Dropout + BatchNorm | 98-98.5% |
| Simple CNN (2-3 conv layers) | 99.0-99.3% |
| CNN + Data Augmentation | 99.3-99.5% |
| State-of-the-art ensembles | 99.7%+ |

In [ ]:
print("PROJECT SUMMARY")
print("="*50)
print(f"1. MNIST dataset: {x_train.shape[0]} train + {x_test.shape[0]} test images")
print(f"2. Model: MLP with 2 hidden layers (128 neurons each)")
print(f"3. Total parameters: {model.count_params():,}")
print(f"4. Optimizer: Adam")
print(f"5. Epochs trained: {len(history.history['loss'])}")
print(f"6. Test accuracy: {test_accuracy:.4f}")
print(f"7. Test loss: {test_loss:.4f}")
print(f"8. Misclassified: {len(misclassified_idx)}/{len(y_test)} samples")